In [4]:
# Imports des bibliothèques
import giskard
import pandas as pd
import requests
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader # Pour lire le PDF et créer la Knowledge Base
from giskard.llm.client.openai import OpenAIClient
from giskard.llm.embeddings.openai import OpenAIEmbedding

# --- CONFIGURATION ---

# 1. Charger la clé API depuis le fichier .env
load_dotenv()
# Giskard trouvera automatiquement la variable d'environnement OPENAI_API_KEY

# 2. Configurer Giskard pour utiliser l'API d'OpenAI pour ses analyses
giskard.llm.set_llm_api("openai")

# 3. Définir les chemins et URL importants
RAG_AGENT_URL = "http://localhost:5001/ask_rag"
#RAG_AGENT_URL = "http://localhost:5002/ask_lightrag"
DOCUMENT_PATH = "./rag_server/document.pdf" # Le document utilisé par l'agent à tester
#DOCUMENT_PATH = "./lightrag_server/document.pdf"
print(f"Configuration terminée. L'agent à évaluer est à l'URL : {RAG_AGENT_URL}")

Configuration terminée. L'agent à évaluer est à l'URL : http://localhost:5001/ask_rag


C:\Users\FAYA COMPUTER\AppData\Local\Temp\ipykernel_22432\3810748639.py:18: DeprecationWarning: set_llm_api is deprecated: https://docs.giskard.ai/en/latest/open_source/setting_up/index.html
  giskard.llm.set_llm_api("openai")


In [5]:
from giskard.rag import AgentAnswer, KnowledgeBase, QATestset, RAGReport, evaluate, generate_testset
print("Chargement du document pour créer la Knowledge Base de Giskard...")

# On charge le document PDF avec le loader de LangChain
loader = PyPDFLoader(DOCUMENT_PATH)
documents = loader.load()

# On prépare les données pour Giskard en les mettant dans un DataFrame pandas
df = pd.DataFrame([doc.page_content for doc in documents], columns=["text"])

# On crée l'objet KnowledgeBase de Giskard
knowledge_base = KnowledgeBase(df)

print(f"Knowledge Base créée avec succès à partir de {len(df)} pages du document.")

Chargement du document pour créer la Knowledge Base de Giskard...
Knowledge Base créée avec succès à partir de 5 pages du document.


In [6]:
print("Génération du jeu de test par Giskard... (cela peut prendre quelques minutes et utilise l'API OpenAI)")

testset = giskard.rag.generate_testset(
    knowledge_base=knowledge_base,
    num_questions=10,  # Générons 30 questions pour un test complet mais rapide
    language='fr',     # La langue des questions à générer
    agent_description="Un chatbot qui répond à des questions sur un projet de thèse sur les ressources marines."
)

# Bonne pratique : sauvegarder le jeu de test pour pouvoir le réutiliser sans le régénérer
testset.save("document.jsonl")

print("Jeu de test de 30 questions généré et sauvegardé !")
# Affichons un aperçu des 5 premières questions
display(testset.to_pandas().head())

Génération du jeu de test par Giskard... (cela peut prendre quelques minutes et utilise l'API OpenAI)
2025-07-10 13:05:15,961 pid:22432 MainThread giskard.rag  INFO     Finding topics in the knowledge base.


C:\Users\FAYA COMPUTER\anaconda3\envs\rag-env\Lib\site-packages\sklearn\utils\deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
C:\Users\FAYA COMPUTER\anaconda3\envs\rag-env\Lib\site-packages\umap\umap_.py:2462: UserWarning: n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1
  warn(


2025-07-10 13:05:50,989 pid:22432 MainThread giskard.rag  INFO     Found 1 topics in the knowledge base.


Generating questions: 100%|████████████████████████████████████████████████████████████| 10/10 [00:53<00:00,  5.30s/it]

Jeu de test de 30 questions généré et sauvegardé !


,question,reference_answer,reference_context,conversation_history,metadata
id,,,,,
2ccb06bd-a849-4e11-a659-e17a403d04d0,Quels sont les thèmes abordés dans les référen...,Les thèmes abordés incluent l'intelligence art...,"Document 4: Aquatique), 13 (Lutte contre l...",[],"{'question_type': 'simple', 'seed_document_id'..."
3fad35d5-3c35-4772-8e02-68f9081651fc,Quel est l'objectif principal de la thèse prop...,L'objectif principal de la thèse est de dévelo...,Document 0: Proposition de Projet de Thèse...,[],"{'question_type': 'simple', 'seed_document_id'..."
d33e9259-da20-467c-ae88-8d686dc7c978,Sous quelles conditions spécifiques et en tena...,Le projet de thèse vise à développer un cadre ...,Document 0: Proposition de Projet de Thèse...,[],"{'question_type': 'complex', 'seed_document_id..."
4e8227c4-c382-4ff6-81d2-c2d8b89915d2,Dans le cadre du projet de thèse sur les resso...,Des techniques post-hoc comme SHAP (SHapley Ad...,"Document 3: santé composite, et vecteur de...",[],"{'question_type': 'complex', 'seed_document_id..."
c7ba719d-499b-4ad9-9255-1d9cd6a79b0c,Dans le cadre de l'étude de la ZEE sud-africai...,"La côte Ouest (système de Benguela, pêche au m...",Document 2: zones côtières subissant une t...,[],"{'question_type': 'distracting element', 'seed..."


In [7]:
def agent_langchain_api_wrapper(question: str, history=None):
    """
    Cette fonction 'enveloppe' notre agent RAG.
    Elle prend une question, appelle l'API de notre conteneur Docker, et retourne la réponse.
    """
    try:
        # On envoie la question au serveur RAG qui tourne sur le port 5001
        response = requests.post(RAG_AGENT_URL, json={"question": question})
        response.raise_for_status() # Lève une erreur en cas de problème (ex: 404, 500)
        
        # On extrait la réponse du JSON
        return response.json().get("answer", "Erreur: clé 'answer' non trouvée.")
    except requests.exceptions.RequestException as e:
        print(f"Erreur de communication avec l'agent RAG : {e}")
        return str(e)

# Testons rapidement notre fonction wrapper
print("Test du pont vers l'agent Docker...")
test_response = agent_langchain_api_wrapper("Quel est le but de cette thèse ?")
print(f"Réponse de l'agent pour le test : {test_response[:100]}...")

Test du pont vers l'agent Docker...
Réponse de l'agent pour le test : Le but de cette thèse est de développer un modèle prédictif intégré, multi-stresseurs et explicable,...


In [8]:
# --- CELLULE D'ÉVALUATION (VERSION FINALE ET SIMPLIFIÉE) ---

from giskard.rag import evaluate

print("Lancement de l'évaluation avec les métriques par défaut de Giskard...")

# On appelle evaluate dans sa forme la plus simple, sans le paramètre 'metrics'.
# Giskard choisira pour nous les métriques standards et compatibles.
report = evaluate(
    agent_langchain_api_wrapper,
    testset=testset,
    knowledge_base=knowledge_base
)

print("Évaluation terminée !")

# On affiche le rapport final
report

Lancement de l'évaluation avec les métriques par défaut de Giskard...


CorrectnessMetric evaluation: 100%|████████████████████████████████████████████████████| 10/10 [00:11<00:00,  1.10s/it]


Évaluation terminée !


Loading BokehJS ...

In [13]:
# --- CELLULE D'ÉVALUATION (VERSION FINALE ET SIMPLIFIÉE) ---

from giskard.rag import evaluate

print("Lancement de l'évaluation avec les métriques par défaut de Giskard...")

# On appelle evaluate dans sa forme la plus simple, sans le paramètre 'metrics'.
# Giskard choisira pour nous les métriques standards et compatibles.
report = evaluate(
    agent_langchain_api_wrapper,
    testset=testset,
    knowledge_base=knowledge_base
)

print("Évaluation terminée !")

# On affiche le rapport final
report

Lancement de l'évaluation avec les métriques par défaut de Giskard...


CorrectnessMetric evaluation: 100%|████████████████████████████████████████████████████| 10/10 [00:12<00:00,  1.28s/it]


Évaluation terminée !


Loading BokehJS ...

In [10]:
# Afficher le rapport interactif complet de Giskard
report

Loading BokehJS ...